# Milestone 2 — Data Preprocessing & Feature Engineering

Building on EDA findings, this notebook:
1. Cleans both datasets (fixes dtypes, removes PII/leakage columns)
2. Handles missing values
3. Engineers new features (tenure groups, avg monthly spend, service count)
4. One-Hot Encodes categorical variables
5. Z-score scales numerical features
6. Splits data 70 / 15 / 15 (train / val / test)
7. Applies SMOTE to the training set only
8. Saves processed datasets to `data/processed/`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

PROCESSED = '../data/processed'
FIGURES   = '../reports/figures'
MODELS    = '../models'
os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(FIGURES, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)

SEED = 42

## 1. Load Raw Data

In [ ]:
telco_raw = pd.read_excel('../data/raw/Telco_customer_churn.xlsx')
india_raw = pd.read_csv('../data/raw/indian_telecom_customers.csv')
print(f'Telco: {telco_raw.shape} | India: {india_raw.shape}')

## 2. Telco Dataset — Cleaning

In [ ]:
telco = telco_raw.copy()

# Drop non-predictive / leakage / PII columns
drop_cols = ['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code',
             'Lat Long', 'Latitude', 'Longitude',
             'Churn Score', 'CLTV',       # derived metrics — leakage risk
             'Churn Reason',              # only available after churn (leakage)
             'Churn Label']               # redundant with Churn Value
telco.drop(columns=drop_cols, inplace=True)

# Fix Total Charges dtype — coerce blanks to NaN, then fill with 0 (tenure=0 rows)
# Note: must chain fillna rather than use inplace=True (pandas 3.0 CoW)
telco['Total Charges'] = pd.to_numeric(telco['Total Charges'], errors='coerce').fillna(0)

print('Telco after cleaning:', telco.shape)
print('Missing values:', telco.isnull().sum().sum())
print('Total Charges NaN:', telco['Total Charges'].isnull().sum())
print('Columns:', telco.columns.tolist())

## 3. Telco Dataset — Feature Engineering

In [ ]:
# Tenure groups: Short (0-12m), Medium (13-36m), Long (37m+)
def tenure_group(months):
    if months <= 12:
        return 'Short'
    elif months <= 36:
        return 'Medium'
    return 'Long'

telco['Tenure Group'] = telco['Tenure Months'].apply(tenure_group)

# Number of add-on services subscribed
service_cols = ['Online Security', 'Online Backup', 'Device Protection',
                'Tech Support', 'Streaming TV', 'Streaming Movies']
telco['Service Count'] = telco[service_cols].apply(lambda row: (row == 'Yes').sum(), axis=1)

# Monthly charge per tenure month (spending intensity)
telco['Charge Per Month'] = np.where(
    telco['Tenure Months'] > 0,
    telco['Total Charges'] / telco['Tenure Months'],
    telco['Monthly Charges']
)

print('Engineered features added: Tenure Group, Service Count, Charge Per Month')
telco[['Tenure Group', 'Service Count', 'Charge Per Month']].describe(include='all')

## 4. Telco Dataset — Encoding & Scaling

In [ ]:
# Separate target
y_telco = telco['Churn Value'].values
X_telco = telco.drop(columns=['Churn Value'])

# Identify column types
cat_cols = X_telco.select_dtypes(include='object').columns.tolist()
num_cols = X_telco.select_dtypes(include=['int64', 'float64']).columns.tolist()
print('Categorical columns:', cat_cols)
print('Numerical columns:', num_cols)

In [ ]:
# One-Hot Encoding — convert entire frame to float to avoid pandas 3.0 CoW NaN issues
X_telco_enc = pd.get_dummies(X_telco, columns=cat_cols, drop_first=False).astype(float)
print('Shape after One-Hot Encoding:', X_telco_enc.shape)
print('All columns are float64:', (X_telco_enc.dtypes == 'float64').all())

In [ ]:
# Train / Val / Test split — 70 / 15 / 15
X_tmp, X_test_t, y_tmp, y_test_t = train_test_split(
    X_telco_enc, y_telco, test_size=0.15, random_state=SEED, stratify=y_telco)

X_train_t, X_val_t, y_train_t, y_val_t = train_test_split(
    X_tmp, y_tmp, test_size=0.15/0.85, random_state=SEED, stratify=y_tmp)

# Explicit copies required for pandas 3.0 Copy-on-Write correctness
X_train_t, X_val_t, X_test_t = X_train_t.copy(), X_val_t.copy(), X_test_t.copy()

print(f'Train : {X_train_t.shape[0]:,} rows  | Churn rate: {y_train_t.mean():.1%}')
print(f'Val   : {X_val_t.shape[0]:,} rows  | Churn rate: {y_val_t.mean():.1%}')
print(f'Test  : {X_test_t.shape[0]:,} rows  | Churn rate: {y_test_t.mean():.1%}')

In [ ]:
# Z-score Standardization — fit on train only
scaler_t = StandardScaler()
num_cols_enc = [c for c in num_cols if c in X_telco_enc.columns]

X_train_t[num_cols_enc] = scaler_t.fit_transform(X_train_t[num_cols_enc])
X_val_t[num_cols_enc]   = scaler_t.transform(X_val_t[num_cols_enc])
X_test_t[num_cols_enc]  = scaler_t.transform(X_test_t[num_cols_enc])

print('Z-score scaling applied to', len(num_cols_enc), 'numerical features')

In [ ]:
# SMOTE — apply to training set only
print('Before SMOTE — Train class distribution:', np.bincount(y_train_t))
smote = SMOTE(random_state=SEED)
X_train_t_sm, y_train_t_sm = smote.fit_resample(X_train_t, y_train_t)
print('After  SMOTE — Train class distribution:', np.bincount(y_train_t_sm))
print(f'Training set grew from {len(y_train_t):,} to {len(y_train_t_sm):,} samples')

In [ ]:
# Visualise class balance before/after SMOTE
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, counts, title in zip(
        axes,
        [np.bincount(y_train_t), np.bincount(y_train_t_sm)],
        ['Before SMOTE', 'After SMOTE']):
    ax.bar(['No Churn (0)', 'Churn (1)'], counts, color=['#4c9be8', '#e84c4c'])
    ax.set_title(f'Telco Training Set — {title}', fontweight='bold')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts):
        ax.text(i, v + 20, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{FIGURES}/11_smote_telco.png', bbox_inches='tight')
plt.show()

## 5. India Dataset — Cleaning & Engineering

In [ ]:
india = india_raw.copy()

# Drop PII and leakage columns
drop_india = ['customer_id', 'name', 'phone', 'pincode',
              'city', 'state',          # very high cardinality geography
              'churn_probability']       # direct target leakage
india.drop(columns=drop_india, inplace=True)

# Binary target
india['churn_bin'] = (india['churn'] == 'Yes').astype(int)
india.drop(columns=['churn'], inplace=True)

print('India after cleaning:', india.shape)
print('Missing values:', india.isnull().sum().sum())

In [ ]:
# Feature engineering
india['tenure_group'] = india['tenure_months'].apply(tenure_group)

# Average monthly spend per tenure
india['charge_per_month'] = np.where(
    india['tenure_months'] > 0,
    india['total_charges'] / india['tenure_months'],
    india['monthly_charges']
)

# Composite engagement score (higher = more engaged = less likely to churn)
india['service_bundle_score'] = india['has_internet'].astype(int) + india['has_ott_bundle'].astype(int)

print('Engineered: tenure_group, charge_per_month, service_bundle_score')

In [ ]:
y_india = india['churn_bin'].values
X_india = india.drop(columns=['churn_bin']).copy()

cat_cols_i = X_india.select_dtypes(include='object').columns.tolist()
num_cols_i  = X_india.select_dtypes(include=['int64', 'float64', 'bool']).columns.tolist()

print('Cat columns:', cat_cols_i)
print('Num columns:', num_cols_i)

In [ ]:
# One-Hot Encode — convert entire frame to float for uniform dtype
X_india_enc = pd.get_dummies(X_india, columns=cat_cols_i, drop_first=False).astype(float)
print('Shape after OHE:', X_india_enc.shape)
print('All columns are float64:', (X_india_enc.dtypes == 'float64').all())

In [ ]:
# Train / Val / Test split
Xi_tmp, Xi_test, yi_tmp, yi_test = train_test_split(
    X_india_enc, y_india, test_size=0.15, random_state=SEED, stratify=y_india)

Xi_train, Xi_val, yi_train, yi_val = train_test_split(
    Xi_tmp, yi_tmp, test_size=0.15/0.85, random_state=SEED, stratify=yi_tmp)

# Explicit copies for pandas 3.0 CoW correctness
Xi_train, Xi_val, Xi_test = Xi_train.copy(), Xi_val.copy(), Xi_test.copy()

print(f'Train : {Xi_train.shape[0]:,} | Val: {Xi_val.shape[0]:,} | Test: {Xi_test.shape[0]:,}')

In [ ]:
# Z-score Scaling — scale all original numerical columns (already float after OHE)
num_cols_i_enc = [c for c in num_cols_i if c in X_india_enc.columns]
scaler_i = StandardScaler()
Xi_train[num_cols_i_enc] = scaler_i.fit_transform(Xi_train[num_cols_i_enc])
Xi_val[num_cols_i_enc]   = scaler_i.transform(Xi_val[num_cols_i_enc])
Xi_test[num_cols_i_enc]  = scaler_i.transform(Xi_test[num_cols_i_enc])

print('Scaled', len(num_cols_i_enc), 'numerical features.')
print('NaN check after scaling — Xi_train:', Xi_train.isnull().sum().sum())

# SMOTE — apply to training set only
print('Before SMOTE:', np.bincount(yi_train))
smote_i = SMOTE(random_state=SEED)
Xi_train_sm, yi_train_sm = smote_i.fit_resample(Xi_train, yi_train)
print('After  SMOTE:', np.bincount(yi_train_sm))

## 6. Save Processed Data & Scalers

In [ ]:
# Telco
X_train_t_sm.to_csv(f'{PROCESSED}/telco_X_train.csv', index=False)
X_val_t.to_csv(f'{PROCESSED}/telco_X_val.csv', index=False)
X_test_t.to_csv(f'{PROCESSED}/telco_X_test.csv', index=False)
pd.Series(y_train_t_sm, name='Churn Value').to_csv(f'{PROCESSED}/telco_y_train.csv', index=False)
pd.Series(y_val_t, name='Churn Value').to_csv(f'{PROCESSED}/telco_y_val.csv', index=False)
pd.Series(y_test_t, name='Churn Value').to_csv(f'{PROCESSED}/telco_y_test.csv', index=False)

# India
Xi_train_sm.to_csv(f'{PROCESSED}/india_X_train.csv', index=False)
Xi_val.to_csv(f'{PROCESSED}/india_X_val.csv', index=False)
Xi_test.to_csv(f'{PROCESSED}/india_X_test.csv', index=False)
pd.Series(yi_train_sm, name='churn').to_csv(f'{PROCESSED}/india_y_train.csv', index=False)
pd.Series(yi_val, name='churn').to_csv(f'{PROCESSED}/india_y_val.csv', index=False)
pd.Series(yi_test, name='churn').to_csv(f'{PROCESSED}/india_y_test.csv', index=False)

# Scalers
with open(f'{MODELS}/scaler_telco.pkl', 'wb') as f:
    pickle.dump(scaler_t, f)
with open(f'{MODELS}/scaler_india.pkl', 'wb') as f:
    pickle.dump(scaler_i, f)

# Feature names
with open(f'{MODELS}/telco_feature_names.pkl', 'wb') as f:
    pickle.dump(X_train_t_sm.columns.tolist(), f)
with open(f'{MODELS}/india_feature_names.pkl', 'wb') as f:
    pickle.dump(Xi_train_sm.columns.tolist(), f)

print('All processed data and scalers saved.')
print(f'Telco train shape : {X_train_t_sm.shape}')
print(f'India train shape : {Xi_train_sm.shape}')

## 7. Preprocessing Summary

| Step | Telco | India |
|---|---|---|
| Columns dropped | 10 (PII + leakage + redundant) | 6 (PII + leakage + geography) |
| dtype fix | `Total Charges` → float | — |
| Engineered features | Tenure Group, Service Count, Charge/Month | Tenure Group, Charge/Month, Service Bundle Score |
| Encoding | One-Hot (all object cols) | One-Hot (all object cols) |
| Scaling | Z-score (fit on train only) | Z-score (fit on train only) |
| Data split | 70/15/15 stratified | 70/15/15 stratified |
| SMOTE | Yes (on train only) | Yes (on train only) |
| Final train size | After SMOTE | After SMOTE |

In [ ]:
print('Milestone 2 — Preprocessing COMPLETE')
print('Processed files saved to:', PROCESSED)
for f in sorted(os.listdir(PROCESSED)):
    print(' ', f)